# 🚀 High-Quality NCERT Educational Artifacts Generator (Kaggle GPU)
### Powered by  (or )

This notebook generates **authentic, pedagogically sound Multiple Choice Questions (MCQs), balanced True/False questions, and revision Flashcards** for all NCERT curriculum chapters.

### Instructions:
1. In the right-hand panel, set **Accelerator** to **GPU T4 x2** (or **GPU P100**).
2. Set **Internet** to **On** (to download the model and dependencies).
3. Upload  to  (or via Kaggle Datasets).
4. Run all cells. In ~15-20 minutes, download the updated !


In [ ]:
# 1. Install & Verify GPU Dependencies
!pip install -q -U transformers torch accelerate bitsandbytes tqdm tabulate

import torch
import os
import gc
import json
import sqlite3
import re
from pathlib import Path
from tqdm.auto import tqdm

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Kaggle settings -> Accelerator -> GPU T4")



In [ ]:
# 2. Load Qwen2.5-7B-Instruct in 4-bit (Fast, Accurate, Low VRAM)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"Loading tokenizer and model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("Model loaded successfully into VRAM!")



In [ ]:
# 3. Locate & Connect to educational_master.db
# Search in common Kaggle paths
possible_paths = [
    Path("/kaggle/working/educational_master.db"),
    Path("/kaggle/input/educational-master-db/educational_master.db"),
    Path("./educational_master.db"),
]

db_path = None
for p in possible_paths:
    if p.exists():
        db_path = p
        break

if db_path is None:
    # If not found, look recursively in /kaggle/
    found = list(Path("/kaggle").rglob("educational_master.db"))
    if found:
        db_path = found[0]

if db_path is None:
    raise FileNotFoundError("Please upload 'educational_master.db' to /kaggle/working/ or as a dataset!")

print(f"Using database at: {db_path}")

# Create working copy if reading from read-only /kaggle/input
working_db = Path("/kaggle/working/educational_master_generated.db")
if not working_db.exists() or working_db.stat().st_size == 0:
    import shutil
    shutil.copy2(str(db_path), str(working_db))
    print(f"Created working copy at: {working_db}")

conn = sqlite3.connect(str(working_db))
cursor = conn.cursor()

# Ensure tables exist
cursor.execute("""
CREATE TABLE IF NOT EXISTS concepts (
    concept_id TEXT PRIMARY KEY,
    chapter_id TEXT,
    name TEXT,
    summary TEXT,
    difficulty TEXT,
    FOREIGN KEY (chapter_id) REFERENCES chapters(chapter_id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS quiz_questions (
    question_id TEXT PRIMARY KEY,
    chapter_id TEXT,
    question_text TEXT,
    option_a TEXT,
    option_b TEXT,
    option_c TEXT,
    option_d TEXT,
    correct_option INTEGER,
    explanation TEXT,
    FOREIGN KEY (chapter_id) REFERENCES chapters(chapter_id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS flashcards (
    card_id TEXT PRIMARY KEY,
    chapter_id TEXT,
    term TEXT,
    definition TEXT,
    FOREIGN KEY (chapter_id) REFERENCES chapters(chapter_id)
);
""")

cursor.execute("SELECT count(*) FROM chapters")
total_chapters = cursor.fetchone()[0]
print(f"Total Chapters to process: {total_chapters}")




In [ ]:
# 4. Pedagogical Prompt & Parser Function
SYSTEM_PROMPT = """You are an expert curriculum educator creating authentic, high-quality educational materials for NCERT textbooks.
Follow these pedagogical rules strictly:
1. CURRICULUM CONCEPTS & SUMMARIES:
   - Identify 3 to 5 core academic concepts that form the backbone of this chapter.
   - For each concept, write a clear, accurate 2-4 sentence textbook explanation covering its principles, formulations, and real-world relevance.
   - Assign appropriate difficulty: 'easy', 'intermediate', or 'advanced'.
   - Avoid storybook filler, characters (e.g. Pihu/Ravi), publisher metadata, or layout artifacts.
2. MULTIPLE CHOICE QUESTIONS (MCQs):
   - 4 distinct, plausible choices (A, B, C, D).
   - Only ONE unambiguously correct answer.
   - 3 realistic distractors (common student misconceptions or related concepts).
   - Step-by-step pedagogical explanation.
3. REVISION FLASHCARDS:
   - Front: Specific question or term (e.g. "What is Coulomb's Law?", "How do you find the perimeter of a rectangle?").
   - Back: Clear, accurate definition, formula, or key principle.

Output valid JSON only:
{
  "concepts": [
    {
      "name": "Concept Name",
      "summary": "Comprehensive textbook explanation...",
      "difficulty": "intermediate"
    }
  ],
  "quizzes": [
    {
      "question": "Question text here?",
      "option_a": "Option A",
      "option_b": "Option B",
      "option_c": "Option C",
      "option_d": "Option D",
      "correct_option": 0,
      "explanation": "Detailed explanation..."
    }
  ],
  "flashcards": [
    {
      "term": "Front prompt / Term",
      "definition": "Back definition / explanation"
    }
  ]
}
"""

def generate_for_chapter(grade, subject, chapter_title, optional_text=""):
    user_prompt = f"""Generate 3-5 authentic Concepts with Summaries, 4 MCQs, and 6 revision Flashcards for:
Grade: {grade}
Subject: {subject}
Chapter Title: {chapter_title}

Reference Text / Keywords (if available):
{optional_text[:2000]}

Return valid JSON adhering strictly to the schema."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1500,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    # Extract JSON block
    match = re.search(r"\{.*\}", response_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass
    return None



In [ ]:
# 5. Batch Generation Loop across all Chapters
cursor.execute("SELECT chapter_id, grade, subject, chapter_title, description FROM chapters ORDER BY grade, subject")
chapters = cursor.fetchall()

LIMIT_CHAPTERS = None  # Set to an integer (e.g. 5) for quick test, or None for all 333 chapters
target_chapters = chapters[:LIMIT_CHAPTERS] if LIMIT_CHAPTERS else chapters

print(f"Beginning GPU generation for {len(target_chapters)} chapters...")

success_count = 0
total_concepts = 0
total_quizzes = 0
total_flashcards = 0

# Clear old entries in working copy
cursor.execute("DELETE FROM concepts;")
cursor.execute("DELETE FROM quiz_questions;")
cursor.execute("DELETE FROM flashcards;")
conn.commit()

pbar = tqdm(target_chapters, desc="Generating Artifacts")
for ch_id, grade, subject, title, desc in pbar:
    data = generate_for_chapter(grade, subject, title, optional_text=desc or "")
    if not data or not isinstance(data, dict):
        continue
        
    concepts = data.get("concepts", [])
    quizzes = data.get("quizzes", [])
    flashcards = data.get("flashcards", [])
    
    for c_idx, c in enumerate(concepts):
        c_id = f"{ch_id}_c_{c_idx}"
        cursor.execute("""
            INSERT INTO concepts (concept_id, chapter_id, name, summary, difficulty)
            VALUES (?, ?, ?, ?, ?)
        """, (
            c_id, ch_id,
            c.get("name", ""),
            c.get("summary", ""),
            c.get("difficulty", "intermediate")
        ))
        total_concepts += 1
    
    for q_idx, q in enumerate(quizzes):
        q_id = f"{ch_id}_q_{q_idx}"
        cursor.execute("""
            INSERT INTO quiz_questions (
                question_id, chapter_id, question_text,
                option_a, option_b, option_c, option_d,
                correct_option, explanation
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            q_id, ch_id, q.get("question", ""),
            q.get("option_a", ""), q.get("option_b", ""),
            q.get("option_c", ""), q.get("option_d", ""),
            int(q.get("correct_option", 0)) % 4,
            q.get("explanation", "")
        ))
        total_quizzes += 1

    for fc_idx, fc in enumerate(flashcards):
        fc_id = f"{ch_id}_fc_{fc_idx}"
        cursor.execute("""
            INSERT INTO flashcards (
                card_id, chapter_id, term, definition
            ) VALUES (?, ?, ?, ?)
        """, (
            fc_id, ch_id,
            fc.get("term", ""),
            fc.get("definition", "")
        ))
        total_flashcards += 1

    success_count += 1
    if success_count % 10 == 0:
        conn.commit()
    pbar.set_postfix({"Concepts": total_concepts, "Quizzes": total_quizzes, "Flashcards": total_flashcards})

conn.commit()
print("
Generation Complete!")
print(f"Successfully processed {success_count}/{len(target_chapters)} chapters.")
print(f"Total Concepts: {total_concepts} | Total Quizzes: {total_quizzes} | Total Flashcards: {total_flashcards}")



In [ ]:
# 6. Quality Audit & Sample Review
from tabulate import tabulate

print("=== Sample Generated MCQs ===")
cursor.execute("""
    SELECT q.question_text, q.option_a, q.option_b, q.option_c, q.option_d, q.correct_option, q.explanation
    FROM quiz_questions q
    ORDER BY RANDOM() LIMIT 3
""")
for r in cursor.fetchall():
    print(f"Q: {r[0]}")
    opts = [('A', r[1]), ('B', r[2]), ('C', r[3]), ('D', r[4])]
    for tag, opt in opts:
        print(f"  {tag}) {opt}")
    print(f"Correct: {'ABCD'[r[5]]}")
    print(f"Explanation: {r[6]}
")

print("=== Sample Generated Flashcards ===")
cursor.execute("SELECT term, definition FROM flashcards ORDER BY RANDOM() LIMIT 3")
for r in cursor.fetchall():
    print(f"Front: {r[0]}")
    print(f"Back:  {r[1]}
")



In [ ]:
# 7. Finalize & Prepare Download
cursor.execute("PRAGMA optimize;")
conn.close()

final_output = Path("/kaggle/working/educational_master.db")
if final_output.exists():
    final_output.unlink()

working_db.rename(final_output)

print("============================================================")
print(f"FINISHED: Database ready at: {final_output}")
print(f"File Size: {final_output.stat().st_size / (1024*1024):.2f} MB")
print("You can now download this file directly from the Kaggle Output pane")
print("and replace pihub/backend/.local_run/artifacts/educational_master.db!")
print("============================================================")

